# VORTEX-AI / CrisisBench: Full-Scale Neural Baselines (Google Colab / Kaggle)

Runs the remaining full-scale GPU baseline jobs (primarily `timegan_fs`: 50,000 iters/phase on NSE; `diffusion_fs` and `quantgan_fs` are skipped automatically if present in the bundle or prior results zip).

### Google Colab Quickstart
1. **Runtime > Change runtime type > GPU** (`T4`, `L4`, or `A100`).
2. Upload `cloud/vortex_colab_bundle.zip` (built via `python cloud/make_bundle.py --colab`, ~5.3 MB) to Google Drive `MyDrive/vortex_ai/` or upload when prompted below.
3. **Run all cells.** Results are saved atomically to `/content/fullscale_results.zip` and synced to Google Drive `MyDrive/vortex_ai/fullscale_results.zip` after **every single completed run** for automatic resume across Colab disconnects.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys, time, zipfile
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'No GPU detected. Enable GPU first (Colab: Runtime > Change runtime type > GPU).'
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

REQUIRED = {
    'torch_geometric': 'torch-geometric',
    'torchsde': 'torchsde',
    'arch': 'arch',
    'hmmlearn': 'hmmlearn',
    'tabulate': 'tabulate',
}
missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    print('Installing missing packages:', missing, flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('All required packages present.')

In [ ]:
KAGGLE = os.path.isdir('/kaggle/working')
USE_DRIVE = True  # Set False on Colab to skip Google Drive mount and use local /content only
OUT_DIR = '/kaggle/working' if KAGGLE else '/content'
STAGE_DIR = '/tmp/vortex_stage' if KAGGLE else '/content/vortex_stage'
DRIVE_DIR = None

if not KAGGLE and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_DIR = '/content/drive/MyDrive/vortex_ai'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print('Google Drive persistence active:', DRIVE_DIR)
    except Exception as exc:
        print('Google Drive mount skipped/unavailable:', exc)
        DRIVE_DIR = None

if os.path.exists(STAGE_DIR):
    shutil.rmtree(STAGE_DIR)
os.makedirs(STAGE_DIR, exist_ok=True)

def find_first(names):
    search_roots = ['/kaggle/input'] if KAGGLE else [p for p in (DRIVE_DIR, '/content') if p]
    for name in names:
        for root in search_roots:
            hits = glob.glob(os.path.join(root, '**', name), recursive=True)
            if hits:
                return hits[0]
    return None

tars = glob.glob('/kaggle/input/**/vortex_bundle.tar', recursive=True) if KAGGLE else []
bundle_zip = find_first(['vortex_colab_bundle.zip', 'vortex_bundle.zip'])
unz = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/crisisbench/queue.py', recursive=True)}, key=len) if KAGGLE else []

if tars:
    print('Extracting tar:', tars[0])
    subprocess.run(['tar', '-xf', tars[0], '-C', STAGE_DIR], check=True)
elif bundle_zip:
    print('Extracting bundle zip:', bundle_zip)
    zipfile.ZipFile(bundle_zip).extractall(STAGE_DIR)
elif unz:
    src_root = os.path.dirname(unz[0])
    print('Copying unpacked tree:', src_root)
    shutil.copytree(src_root, os.path.join(STAGE_DIR, 'vortex'), dirs_exist_ok=True)
elif not KAGGLE:
    from google.colab import files
    print('Upload vortex_colab_bundle.zip (or vortex_bundle.zip), and optionally fullscale_results.zip:')
    uploaded = files.upload()
    for name in uploaded:
        if 'bundle' in name and name.endswith('.zip'):
            zipfile.ZipFile(name).extractall(STAGE_DIR)
else:
    raise RuntimeError('No bundle found under /kaggle/input')

REPO = os.path.join(STAGE_DIR, 'vortex')
if not os.path.isdir(os.path.join(REPO, 'crisisbench')):
    alt = glob.glob(os.path.join(STAGE_DIR, '**/crisisbench'), recursive=True)
    assert alt, f'crisisbench not found under {STAGE_DIR}'
    REPO = os.path.dirname(alt[0])

# Auto-resume from prior fullscale_results.zip (Drive first, then /content or /kaggle/input)
prev = find_first(['fullscale_results.zip'])
if prev:
    print('Resuming from existing results zip:', prev)
    zipfile.ZipFile(prev).extractall(REPO)

os.chdir(REPO)
for gen in ('diffusion_fs', 'quantgan_fs', 'timegan_fs'):
    done = len(glob.glob(os.path.join(REPO, 'results', 'runs', 'nse', gen, '*', 'seed*', 'metrics.json')))
    print(f'Pre-seeded {gen:13s}: {done}/14 metrics.json present')

In [ ]:
# Colab default: run ONLY remaining timegan_fs jobs (diffusion_fs & quantgan_fs already finished on Kaggle).
# Set COLAB_GENS = 'diffusion,quantgan,timegan' if you want fullscale.py to check/run all three.
SHARD_MODE = 'all'
WORKERS = 4
THREADS = 1
COLAB_GENS = 'timegan'
HOURS = 11.0 if KAGGLE else 10.0
OUT_ZIP = os.path.join(OUT_DIR, 'fullscale_results.zip')
DRIVE_ZIP = os.path.join(DRIVE_DIR, 'fullscale_results.zip') if DRIVE_DIR else None

os.environ['CRISISBENCH_DEVICE'] = 'cuda'
os.environ['PYTHONWARNINGS'] = 'ignore'

if KAGGLE:
    cmd = [
        sys.executable, '-m', 'cloud.kaggle_runner',
        '--mode', SHARD_MODE,
        '--workers', str(WORKERS),
        '--threads', str(THREADS),
        '--hours', str(HOURS),
        '--out', OUT_ZIP,
    ]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=REPO)
else:
    cmd = [
        sys.executable, '-u', '-m', 'crisisbench.fullscale',
        '--gens', COLAB_GENS,
        '--hours', str(HOURS),
        '--out', OUT_ZIP,
    ]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO)
    last_mtime = 0.0
    while proc.poll() is None:
        if DRIVE_ZIP and os.path.exists(OUT_ZIP) and not os.path.exists(os.path.splitext(OUT_ZIP)[0] + '.tmp'):
            mt = os.path.getmtime(OUT_ZIP)
            if mt > last_mtime and zipfile.is_zipfile(OUT_ZIP):
                tmp = DRIVE_ZIP + '.tmp'
                shutil.copy2(OUT_ZIP, tmp)
                os.replace(tmp, DRIVE_ZIP)
                last_mtime = mt
                print(f'[drive-sync] updated {DRIVE_ZIP} ({os.path.getsize(DRIVE_ZIP)/1e6:.2f} MB)', flush=True)
        time.sleep(15)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    if DRIVE_ZIP and os.path.exists(OUT_ZIP):
        tmp = DRIVE_ZIP + '.tmp'
        shutil.copy2(OUT_ZIP, tmp)
        os.replace(tmp, DRIVE_ZIP)
        print(f'[drive-sync] final {DRIVE_ZIP} ({os.path.getsize(DRIVE_ZIP)/1e6:.2f} MB)', flush=True)

In [ ]:
if os.path.exists(OUT_ZIP):
    print(f'Output bundle ready: {OUT_ZIP} ({os.path.getsize(OUT_ZIP) / 1e6:.2f} MB)')
    for gen in ('diffusion_fs', 'quantgan_fs', 'timegan_fs'):
        done = len(glob.glob(os.path.join(REPO, 'results', 'runs', 'nse', gen, '*', 'seed*', 'metrics.json')))
        print(f'  {gen:13s}: {done}/14 metrics.json')
    if not KAGGLE:
        from google.colab import files
        files.download(OUT_ZIP)
else:
    print('No output zip produced.')